# Linear Algebra for ML — Exercises

Companion to the note [Linear Algebra for ML](Linear%20Algebra%20for%20ML.md).

Practise norms and dot products, rank and null spaces, PSD matrices, eigendecomposition, the SVD and Eckart–Young, least squares with the pseudo-inverse, projections and the trace/determinant identities: first by hand, then from scratch in NumPy.

**19 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.metrics.pairwise import rbf_kernel
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Rank, determinant, invertibility
*🧠 Concept · ★☆☆*

For a square $n\times n$ matrix $A$ the note says *invertible ⇔ full rank ⇔ $\det A\neq 0$*. Add two more equivalent
statements involving (a) the null space and (b) the eigenvalues. Then decide, without computing anything heavy, whether
these are invertible:

1. $\begin{pmatrix}1&2\\2&4\end{pmatrix}$  2. $\begin{pmatrix}2&0\\0&-3\end{pmatrix}$  3. a $5\times 5$ matrix with a column of zeros  4. $X^\top X$ where $X$ is $3\times 10$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\det A = \prod_i \lambda_i$. And $\operatorname{rank}(X^\top X) = \operatorname{rank}(X) \le \min(\text{rows}, \text{cols})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Equivalent: (a) $\mathcal N(A)=\{0\}$ ($Ax=0$ only for $x=0$); (b) no eigenvalue equals $0$ (since $\det A=\prod\lambda_i$).

1. Not invertible: row 2 = 2·row 1, rank 1, $\det = 4-4 = 0$.
2. Invertible: diagonal with non-zero entries, $\det=-6$.
3. Not invertible: $Ae_j = 0$ for the zero column $j$, so the null space is non-trivial.
4. Not invertible: $X^\top X$ is $10\times10$ but has rank $\le 3$. This is the "more features than samples" case where the
normal equations break down ([[Linear Regression]]).

</details>

### Exercise 2 · Why covariance and Gram matrices are PSD
*🧠 Concept · ★★☆*

Show that for any data matrix $X\in\mathbb R^{N\times d}$ the Gram matrix $X^\top X$ is positive semi-definite, and conclude
that the sample covariance $\frac1N X_c^\top X_c$ (with centred $X_c$) is PSD. When is $X^\top X$ positive *definite*?
Why does this matter for kernel matrices ([[Kernel Methods]])?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $v^\top X^\top X v$ as a squared norm.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$v^\top X^\top X v = (Xv)^\top (Xv) = \lVert Xv\rVert_2^2 \ge 0$ for all $v$, so $X^\top X\succeq 0$; scaling by $\frac1N>0$
keeps this. It is positive definite iff $Xv\neq 0$ for all $v\neq0$, i.e. iff $X$ has full column rank (linearly independent
features, $N\ge d$).

A valid kernel $k(x,x')=\langle\phi(x),\phi(x')\rangle$ gives $K=\Phi\Phi^\top$, PSD by the same argument. PSD-ness guarantees that
kernel methods (SVM dual, GP) solve convex problems and that $K+\lambda I$ is invertible for $\lambda>0$.

</details>

### Exercise 3 · SVD versus eigendecomposition
*🧠 Concept · ★★☆*

Let $X = U\Sigma V^\top$ be the (thin) SVD of a centred data matrix. Express the eigendecomposition of $X^\top X$ and of
$XX^\top$ in terms of $U,\Sigma,V$. Use this to explain why [[PCA]] can be computed either from the covariance matrix or
directly from the SVD of $X$, and why the SVD route is numerically preferred.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Multiply out $X^\top X = V\Sigma U^\top U \Sigma V^\top$ and use $U^\top U = I$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$X^\top X = V\Sigma^2 V^\top$ and $XX^\top = U\Sigma^2U^\top$. So the right singular vectors $V$ are the eigenvectors of the
(scaled) covariance $\frac1N X^\top X$, with eigenvalues $\sigma_i^2/N$; the left singular vectors are eigenvectors of the
Gram matrix. PCA directions are the columns of $V$, the scores are $XV = U\Sigma$.

Forming $X^\top X$ squares the condition number ($\kappa(X^\top X)=\kappa(X)^2$), which loses about half the significant digits
for small singular values. The SVD works on $X$ directly and avoids this.

</details>

### Exercise 4 · Under-determined least squares
*🧠 Concept · ★★★*

If $N<d$, $\lVert Xw-y\rVert^2$ has infinitely many minimisers (all with zero error, if $X$ has full row rank). Which one does
$w = X^+y$ pick? Why is that a sensible choice, and which regulariser does it correspond to in the limit?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Any solution is $w = w_0 + n$ with $n\in\mathcal N(X)$. Where does $X^+y$ live? Compare with ridge as $\lambda\to 0^+$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$X^+ y = V\Sigma^{-1}U^\top y$ lies in the row space of $X$ (span of the columns of $V$), which is orthogonal to $\mathcal N(X)$.
Adding any null-space vector $n$ keeps $Xw$ fixed but increases $\lVert w\rVert^2 = \lVert X^+y\rVert^2 + \lVert n\rVert^2$.
So the pseudo-inverse returns the **minimum-norm** interpolating solution. It equals
$\lim_{\lambda\to0^+}(X^\top X+\lambda I)^{-1}X^\top y$, the limit of ridge regression. This implicit bias towards small
norm is part of why heavily over-parameterised models can still generalise.

</details>

## Part B · By hand

### Exercise 5 · Norms and angles
*✍️ By hand · ★☆☆*

Let $a = (1, 2, 2)^\top$ and $b = (0, 3, 4)^\top$. Compute $\lVert a\rVert_1$, $\lVert a\rVert_2$, $\lVert a\rVert_\infty$,
$a^\top b$ and $\cos\theta$ between $a$ and $b$.

In [ ]:
a_vec = np.array([1., 2., 2.]); b_vec = np.array([0., 3., 4.])
norm1 = None; norm2 = None; norminf = None
dot_ab = None; cos_ab = None

check('||a||_1', norm1, np.linalg.norm(a_vec, 1))
check('||a||_2', norm2, np.linalg.norm(a_vec))
check('||a||_inf', norminf, np.linalg.norm(a_vec, np.inf))
check('a.b', dot_ab, np.dot(a_vec, b_vec))
check('cos theta', cos_ab, np.dot(a_vec, b_vec) / np.linalg.norm(a_vec) / np.linalg.norm(b_vec))

<details>
<summary><b>💡 Hint</b></summary>

$\cos\theta = \frac{a^\top b}{\lVert a\rVert_2\lVert b\rVert_2}$; $\lVert b\rVert_2 = 5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\lVert a\rVert_1 = 5$, $\lVert a\rVert_2 = \sqrt{1+4+4}=3$, $\lVert a\rVert_\infty = 2$, $a^\top b = 0+6+8 = 14$,
$\cos\theta = 14/(3\cdot 5) = 14/15 \approx 0.933$.

```python
a_vec = np.array([1., 2., 2.]); b_vec = np.array([0., 3., 4.])
norm1 = 5; norm2 = 3; norminf = 2
dot_ab = 14; cos_ab = 14 / 15
```

</details>

### Exercise 6 · Products, transposes, traces
*✍️ By hand · ★☆☆*

With $A = \begin{pmatrix}1&2\\0&1\\3&-1\end{pmatrix}$ and $B = \begin{pmatrix}2&1&0\\1&0&1\end{pmatrix}$, compute $AB$ ($3\times3$)
and $BA$ ($2\times2$). Verify $\mathrm{tr}(AB)=\mathrm{tr}(BA)$ even though the matrices have different sizes, and check
$(AB)^\top = B^\top A^\top$ on one entry.

In [ ]:
A_m = np.array([[1., 2.], [0., 1.], [3., -1.]]); B_m = np.array([[2., 1., 0.], [1., 0., 1.]])
AB_hand = None   # 3x3 nested list
BA_hand = None   # 2x2 nested list

check('AB', AB_hand, A_m @ B_m)
check('BA', BA_hand, B_m @ A_m)

<details>
<summary><b>💡 Hint</b></summary>

$(AB)_{ij} = \sum_k A_{ik}B_{kj}$: row $i$ of $A$ dotted with column $j$ of $B$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$AB = \begin{pmatrix}4&1&2\\1&0&1\\5&3&-1\end{pmatrix}$, $BA = \begin{pmatrix}2&5\\4&1\end{pmatrix}$.
$\mathrm{tr}(AB) = 4+0-1 = 3 = 2+1 = \mathrm{tr}(BA)$. ✔ E.g. $(AB)^\top_{13} = (AB)_{31} = 5$ and
$(B^\top A^\top)_{13} = \sum_k B_{k1}A_{3k} = 2\cdot3 + 1\cdot(-1) = 5$. ✔

```python
A_m = np.array([[1., 2.], [0., 1.], [3., -1.]]); B_m = np.array([[2., 1., 0.], [1., 0., 1.]])
AB_hand = [[4, 1, 2], [1, 0, 1], [5, 3, -1]]
BA_hand = [[2, 5], [4, 1]]
```

</details>

### Exercise 7 · Eigendecomposition of a symmetric 2×2
*✍️ By hand · ★☆☆*

Find the eigenvalues and orthonormal eigenvectors of $S = \begin{pmatrix}2&1\\1&2\end{pmatrix}$, write $S = Q\Lambda Q^\top$, and
verify $\mathrm{tr}\,S = \sum\lambda_i$ and $\det S = \prod\lambda_i$. Is $S$ positive definite?

In [ ]:
S_sym = np.array([[2., 1.], [1., 2.]])
eigvals_hand = None   # sorted ascending, e.g. [l1, l2]

check('eigenvalues', eigvals_hand, np.linalg.eigvalsh(S_sym))

<details>
<summary><b>💡 Hint</b></summary>

Solve $\det(S-\lambda I) = (2-\lambda)^2 - 1 = 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(2-\lambda)^2 = 1 \Rightarrow \lambda\in\{1, 3\}$. For $\lambda=3$: $v=(1,1)/\sqrt2$; for $\lambda=1$: $v=(1,-1)/\sqrt2$.
$Q = \frac1{\sqrt2}\begin{pmatrix}1&1\\-1&1\end{pmatrix}$ (columns for $\lambda=1,3$), $\Lambda=\mathrm{diag}(1,3)$.
$\mathrm{tr}=4=1+3$, $\det = 3 = 1\cdot 3$. Both eigenvalues are positive, so $S\succ0$.

```python
S_sym = np.array([[2., 1.], [1., 2.]])
eigvals_hand = [1, 3]
```

</details>

### Exercise 8 · Rank and null space
*✍️ By hand · ★★☆*

Let $M = \begin{pmatrix}1&2&3\\2&4&6\\1&0&1\end{pmatrix}$. Find its rank, $\det M$, and a non-zero vector spanning its null space.
What is $\dim\mathcal N(M)$ (rank–nullity)?

In [ ]:
M_rank = np.array([[1., 2., 3.], [2., 4., 6.], [1., 0., 1.]])
rank_hand = None
null_vec = None   # a non-zero vector v with M v = 0

check('rank', rank_hand, np.linalg.matrix_rank(M_rank))
if null_vec is None:
    check('M v = 0 and v != 0', None, True)
else:
    _v = np.asarray(null_vec, float)
    check('M v = 0 and v != 0', np.allclose(M_rank @ _v, 0) and np.linalg.norm(_v) > 0, True)

<details>
<summary><b>💡 Hint</b></summary>

Row 2 is twice row 1. Solve $x+2y+3z=0$ and $x+z=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Rows 1 and 2 are dependent, rows 1 and 3 are not, so $\operatorname{rank} M = 2$ and $\det M=0$.
From $x+z=0$: $x=-z$; then $-z+2y+3z=0\Rightarrow y=-z$. With $z=1$: $v=(-1,-1,1)^\top$.
Rank–nullity: $\dim\mathcal N(M) = 3-2 = 1$.

```python
M_rank = np.array([[1., 2., 3.], [2., 4., 6.], [1., 0., 1.]])
rank_hand = 2
null_vec = [-1, -1, 1]
```

</details>

### Exercise 9 · Projection onto a column space
*✍️ By hand · ★★☆*

With $X = \begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$ and $y = (1,2,2)^\top$, compute the projection $\hat y = Py$ where
$P = X(X^\top X)^{-1}X^\top$. Then prove in general that $P^2 = P$ and $P^\top = P$, and that the residual $y-Py$ is
orthogonal to every column of $X$.

In [ ]:
X_p = np.array([[1., 0.], [1., 1.], [1., 2.]]); y_p = np.array([1., 2., 2.])
yhat_proj = None   # the projected vector P y

check('P y', yhat_proj, X_p @ np.linalg.lstsq(X_p, y_p, rcond=None)[0])

<details>
<summary><b>💡 Hint 1</b></summary>

$X^\top X = \begin{pmatrix}3&3\\3&5\end{pmatrix}$, $X^\top y = (5, 6)^\top$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$Py = Xw$ with $w = (X^\top X)^{-1}X^\top y$; a $2\times2$ inverse is $\frac{1}{ad-bc}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(X^\top X)^{-1} = \frac16\begin{pmatrix}5&-3\\-3&3\end{pmatrix}$, $w = \frac16(25-18,\,-15+18) = (7/6,\ 1/2)$,
so $\hat y = Xw = (7/6,\ 5/3,\ 13/6) \approx (1.167, 1.667, 2.167)$.

$P^2 = X(X^\top X)^{-1}X^\top X(X^\top X)^{-1}X^\top = X(X^\top X)^{-1}X^\top = P$; $P^\top = P$ because $(X^\top X)^{-1}$ is symmetric.
$X^\top(y - Py) = X^\top y - X^\top X(X^\top X)^{-1}X^\top y = 0$: the residual is orthogonal to col$(X)$, which is exactly
the normal equations.

```python
X_p = np.array([[1., 0.], [1., 1.], [1., 2.]]); y_p = np.array([1., 2., 2.])
yhat_proj = [7/6, 5/3, 13/6]
```

</details>

### Exercise 10 · SVD and the best rank-1 approximation
*✍️ By hand · ★★☆*

Let $A = \begin{pmatrix}3&0\\4&5\end{pmatrix}$. Compute its singular values from the eigenvalues of $A^\top A$. By Eckart–Young,
what is the squared Frobenius error $\lVert A - A_1\rVert_F^2$ of the best rank-1 approximation? (Bonus: find $A_1$.)

In [ ]:
A_svd = np.array([[3., 0.], [4., 5.]])
sing_vals = None    # descending
err_rank1_sq = None

check('singular values', sing_vals, np.linalg.svd(A_svd, compute_uv=False))
_U, _s, _Vt = np.linalg.svd(A_svd)
check('rank-1 error^2', err_rank1_sq, np.sum((A_svd - _s[0] * np.outer(_U[:, 0], _Vt[0])) ** 2))

<details>
<summary><b>💡 Hint 1</b></summary>

$A^\top A = \begin{pmatrix}25&20\\20&25\end{pmatrix}$, eigenvalues $25\pm20$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Eckart–Young: $\lVert A-A_k\rVert_F^2 = \sum_{i>k}\sigma_i^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Eigenvalues of $A^\top A$: $45$ and $5$, so $\sigma_1 = \sqrt{45} = 3\sqrt5\approx 6.708$, $\sigma_2=\sqrt5\approx2.236$.
$\lVert A-A_1\rVert_F^2 = \sigma_2^2 = 5$. With $v_1 = (1,1)/\sqrt2$ and $u_1 = Av_1/\sigma_1 = (3,9)/\sqrt{90} = (1,3)/\sqrt{10}$:
$A_1 = \sigma_1 u_1v_1^\top = \begin{pmatrix}1.5&1.5\\4.5&4.5\end{pmatrix}$; indeed $A-A_1 = \begin{pmatrix}1.5&-1.5\\-0.5&0.5\end{pmatrix}$
has squared norm $2.25+2.25+0.25+0.25 = 5$. ✔

```python
A_svd = np.array([[3., 0.], [4., 5.]])
sing_vals = [np.sqrt(45), np.sqrt(5)]
err_rank1_sq = 5
```

</details>

### Exercise 11 · Sherman–Morrison (rank-1 Woodbury)
*✍️ By hand · ★★★*

The Woodbury lemma with $U=u$, $C=1$, $V=v^\top$ becomes Sherman–Morrison:
$$(A + uv^\top)^{-1} = A^{-1} - \frac{A^{-1}uv^\top A^{-1}}{1 + v^\top A^{-1}u}.$$
Verify it by multiplying the right-hand side by $A+uv^\top$. What is the cost of updating an inverse this way compared with
re-inverting, and where is this useful in ML (think online least squares)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Let $B = A^{-1}$ and $c = 1+v^\top Bu$ (a scalar). Expand $(A+uv^\top)(B - Buv^\top B/c)$ and group the $u\,(\dots)\,v^\top B$ terms.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(A+uv^\top)\left(B - \frac{Buv^\top B}{c}\right) = I + uv^\top B - \frac{uv^\top B + u(v^\top Bu)v^\top B}{c}
= I + uv^\top B - \frac{u(1+v^\top Bu)v^\top B}{c} = I + uv^\top B - uv^\top B = I.$ ✔

The update costs $O(d^2)$ (matrix–vector products) versus $O(d^3)$ for a fresh inverse. In recursive least squares each new
sample adds $x x^\top$ to $X^\top X$, so $(X^\top X)^{-1}$ can be updated in $O(d^2)$ per sample; the same trick gives fast
leave-one-out CV for ridge and is used in Gaussian-process and Kalman-filter updates.

</details>

## Part C · Code from scratch

### Exercise 12 · Matrix product from the definition
*💻 Code · ★☆☆*

Implement `matmul(A, B)` using only the definition $(AB)_{ij}=\sum_k A_{ik}B_{kj}$ (loops, or one loop plus a dot product).
Raise a `ValueError` if the inner dimensions do not match. Compare with `A @ B` on random matrices.

In [ ]:
def matmul(A, B):
    # TODO
    return None

A_r = rng.normal(size=(4, 3)); B_r = rng.normal(size=(3, 5))
C_mine = matmul(A_r, B_r)

check('matmul', C_mine, A_r @ B_r)

<details>
<summary><b>💡 Hint</b></summary>

Allocate `C = np.zeros((n, p))` and fill `C[i, j] = sum(A[i, k] * B[k, j] for k in range(m))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The triple loop is $O(nmp)$ like BLAS, but BLAS is orders of magnitude faster thanks to blocking, SIMD and cache reuse: always use `@` in practice.

```python
def matmul(A, B):
    n, m = A.shape
    m2, p = B.shape
    if m != m2:
        raise ValueError(f"inner dimensions differ: {m} vs {m2}")
    C = np.zeros((n, p))
    for i in range(n):
        for j in range(p):
            C[i, j] = sum(A[i, k] * B[k, j] for k in range(m))
    return C

A_r = rng.normal(size=(4, 3)); B_r = rng.normal(size=(3, 5))
C_mine = matmul(A_r, B_r)
```

</details>

### Exercise 13 · PSD test and kernel matrices
*💻 Code · ★☆☆*

Write `is_psd(A, tol=1e-10)` that returns `True` iff $A$ is symmetric and all eigenvalues are $\ge -\text{tol}$.
Test it on (a) an RBF kernel matrix of 30 random points, (b) $X^\top X$ for a random $X$, (c) $\begin{pmatrix}1&2\\2&1\end{pmatrix}$.

In [ ]:
def is_psd(A, tol=1e-10):
    # TODO
    return None

X_k = rng.normal(size=(30, 2))
K_rbf = rbf_kernel(X_k, gamma=0.5)
psd_results = [is_psd(K_rbf), is_psd(X_k.T @ X_k), is_psd(np.array([[1., 2.], [2., 1.]]))]

if psd_results[0] is None:
    check('PSD results', None, True)
else:
    check('PSD results', psd_results, [True, True, False])

<details>
<summary><b>💡 Hint</b></summary>

Use `np.linalg.eigvalsh` (for symmetric matrices) after checking `np.allclose(A, A.T)`. Matrix (c) has eigenvalues $3$ and $-1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Kernel and Gram matrices are PSD (Exercise 2); $\begin{pmatrix}1&2\\2&1\end{pmatrix}$ has eigenvalue $-1$ so $x=(1,-1)$ gives $x^\top Ax=-2<0$. Use a small tolerance: round-off can make tiny eigenvalues slightly negative.

```python
def is_psd(A, tol=1e-10):
    A = np.asarray(A, float)
    if not np.allclose(A, A.T):
        return False
    return bool(np.linalg.eigvalsh(A).min() >= -tol)

X_k = rng.normal(size=(30, 2))
K_rbf = rbf_kernel(X_k, gamma=0.5)
psd_results = [is_psd(K_rbf), is_psd(X_k.T @ X_k), is_psd(np.array([[1., 2.], [2., 1.]]))]
```

</details>

### Exercise 14 · Gram–Schmidt QR
*💻 Code · ★★☆*

Implement `gram_schmidt(A)` returning $Q$ (orthonormal columns) and upper-triangular $R$ with $A = QR$, for a full-column-rank
$A\in\mathbb R^{n\times d}$. Use the *modified* Gram–Schmidt variant. Check $Q^\top Q = I$ and $QR = A$.

In [ ]:
def gram_schmidt(A):
    # TODO: return Q (n x d), R (d x d)
    return None, None

A_qr = rng.normal(size=(6, 4))
Q_gs, R_gs = gram_schmidt(A_qr)

if Q_gs is None:
    check('QR', None, True)
else:
    check('Q^T Q = I', Q_gs.T @ Q_gs, np.eye(4))
    check('QR = A', Q_gs @ R_gs, A_qr)
    check('R upper triangular', np.allclose(np.tril(R_gs, -1), 0), True)
    check('|R| matches numpy', np.abs(R_gs), np.abs(np.linalg.qr(A_qr)[1]))

<details>
<summary><b>💡 Hint</b></summary>

For column $j$: start from $v = a_j$; for each earlier $q_i$, set $R_{ij} = q_i^\top v$ and $v \leftarrow v - R_{ij}q_i$; then $R_{jj} = \lVert v\rVert$, $q_j = v/R_{jj}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Modified GS subtracts projections from the *updated* $v$, which is much more stable than classical GS. NumPy's QR uses Householder reflections and may flip signs of columns of $Q$ / rows of $R$, hence the comparison in absolute value. QR gives least squares as $w = R^{-1}Q^\top y$ without forming $X^\top X$.

```python
def gram_schmidt(A):
    n, d = A.shape
    Q = np.zeros((n, d)); R = np.zeros((d, d))
    for j in range(d):
        v = A[:, j].astype(float).copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ v
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

A_qr = rng.normal(size=(6, 4))
Q_gs, R_gs = gram_schmidt(A_qr)
```

</details>

### Exercise 15 · Power iteration
*💻 Code · ★★☆*

Implement `power_iteration(S, iters=500)` returning the dominant eigenvalue and unit eigenvector of a symmetric PSD matrix
(use the Rayleigh quotient $v^\top Sv$). Apply it to the covariance matrix of the given data. How does the convergence speed
depend on $\lambda_2/\lambda_1$?

In [ ]:
def power_iteration(S, iters=500):
    # TODO: return (eigenvalue, eigenvector)
    return None, None

X_pi = rng.normal(size=(200, 5)) @ np.diag([3., 2., 1., 0.5, 0.1])
S_pi = np.cov(X_pi, rowvar=False)
lam_pi, v_pi = power_iteration(S_pi)

_w, _V = np.linalg.eigh(S_pi)
check('top eigenvalue', lam_pi, _w[-1])
check('|<v, v_true>| = 1', None if v_pi is None else abs(v_pi @ _V[:, -1]), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Repeat `v = S @ v; v /= np.linalg.norm(v)` from a random start. The error shrinks like $(\lambda_2/\lambda_1)^t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Writing $v_0 = \sum_i c_i q_i$, $S^tv_0 = \sum_i c_i\lambda_i^t q_i$, which is dominated by $q_1$ at rate $|\lambda_2/\lambda_1|^t$. A small eigengap means slow convergence. Deflation ($S \leftarrow S - \lambda_1 q_1q_1^\top$) gives the next component, a simple way to get the first few PCs ([[PCA]]).

```python
def power_iteration(S, iters=500):
    v = np.random.default_rng(1).normal(size=S.shape[0])
    v /= np.linalg.norm(v)
    for _ in range(iters):
        v = S @ v
        v /= np.linalg.norm(v)
    return v @ S @ v, v

X_pi = rng.normal(size=(200, 5)) @ np.diag([3., 2., 1., 0.5, 0.1])
S_pi = np.cov(X_pi, rowvar=False)
lam_pi, v_pi = power_iteration(S_pi)
```

</details>

### Exercise 16 · Low-rank approximation (Eckart–Young)
*💻 Code · ★★☆*

Write `low_rank(A, k)` returning the best rank-$k$ approximation via the SVD. For a random $20\times 15$ matrix and
$k=1,\dots,15$ compute the Frobenius errors in `errs_lr` and confirm they equal $\sqrt{\sum_{i>k}\sigma_i^2}$. Store the rank of
`low_rank(A, 3)` in `rank_of_A3`.

In [ ]:
def low_rank(A, k):
    # TODO
    return None

A_lr = rng.normal(size=(20, 15))
errs_lr = None       # list of 15 Frobenius errors, k = 1..15
rank_of_A3 = None

_s = np.linalg.svd(A_lr, compute_uv=False)
check('errors = tail singular values', errs_lr, [np.sqrt(np.sum(_s[k:] ** 2)) for k in range(1, 16)])
check('rank of A_3', rank_of_A3, 3)

<details>
<summary><b>💡 Hint</b></summary>

`U, s, Vt = np.linalg.svd(A, full_matrices=False)`; then `U[:, :k] * s[:k] @ Vt[:k]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The error decreases to $0$ at $k=\operatorname{rank}(A)=15$. Truncated SVD is used in PCA, latent semantic analysis, compressing weight matrices and low-rank adapters.

```python
def low_rank(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return (U[:, :k] * s[:k]) @ Vt[:k]

A_lr = rng.normal(size=(20, 15))
errs_lr = [np.linalg.norm(A_lr - low_rank(A_lr, k)) for k in range(1, 16)]
rank_of_A3 = np.linalg.matrix_rank(low_rank(A_lr, 3))
```

</details>

### Exercise 17 · Pseudo-inverse from the SVD
*💻 Code · ★★☆*

Implement `pinv_svd(X, rcond=1e-12)` as $X^+ = V\Sigma^{+}U^\top$, where $\Sigma^+$ inverts singular values above
`rcond * max(s)` and zeros the rest. Then solve an **under-determined** problem ($N=5$ samples, $d=8$ features) and verify that
your solution (a) fits the data exactly and (b) has a smaller norm than another exact solution $w + n$ with $n\in\mathcal N(X)$.

In [ ]:
def pinv_svd(X, rcond=1e-12):
    # TODO
    return None

X_u = rng.normal(size=(5, 8)); y_u = rng.normal(size=5)
Xp = pinv_svd(X_u)
w_min = None if Xp is None else Xp @ y_u

check('pinv', Xp, np.linalg.pinv(X_u))
if w_min is None:
    check('exact fit and minimum norm', None, True)
else:
    _n = np.linalg.svd(X_u)[2][-1]          # a null-space direction of X_u
    check('exact fit and minimum norm', np.allclose(X_u @ w_min, y_u) and np.linalg.norm(w_min) < np.linalg.norm(w_min + _n), True)

<details>
<summary><b>💡 Hint</b></summary>

`U, s, Vt = np.linalg.svd(X, full_matrices=False)`; `s_inv = np.where(s > rcond * s.max(), 1/s, 0)`; return `Vt.T * s_inv @ U.T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

See Exercise 4: $X^+y$ lies in the row space, and any null-space component only adds norm. Thresholding small singular values is what makes `pinv`/`lstsq` robust to rank deficiency.

```python
def pinv_svd(X, rcond=1e-12):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    s_inv = np.where(s > rcond * s.max(), 1.0 / s, 0.0)
    return (Vt.T * s_inv) @ U.T

X_u = rng.normal(size=(5, 8)); y_u = rng.normal(size=5)
Xp = pinv_svd(X_u)
w_min = None if Xp is None else Xp @ y_u
```

</details>

### Exercise 18 · Trace and determinant identities numerically
*💻 Code · ★☆☆*

For random matrices verify numerically: $\mathrm{tr}(AB)=\mathrm{tr}(BA)$ (with $A$ $3\times5$, $B$ $5\times3$),
$\mathrm{tr}(S)=\sum\lambda_i$ and $\det S=\prod\lambda_i$ for a random symmetric $S$, and $\det(AB)=\det A\det B$ for square $A,B$.
Store the four boolean results in `identities_ok` (a list).

In [ ]:
identities_ok = None   # [tr(AB)=tr(BA), tr=sum eig, det=prod eig, det(AB)=detA detB]

check('all identities hold', None if identities_ok is None else all(identities_ok) and len(identities_ok) == 4, True)

<details>
<summary><b>💡 Hint</b></summary>

Make a random symmetric matrix as `(M + M.T) / 2`. Use `np.isclose` for comparisons.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All hold up to round-off. The trace identity extends to cyclic permutations $\mathrm{tr}(ABC)=\mathrm{tr}(CAB)$, which is the workhorse of matrix-calculus derivations ([[Matrix Calculus Cheatsheet]]).

```python
A1 = rng.normal(size=(3, 5)); B1 = rng.normal(size=(5, 3))
M1 = rng.normal(size=(4, 4)); S1 = (M1 + M1.T) / 2
lam1 = np.linalg.eigvalsh(S1)
P1 = rng.normal(size=(4, 4)); P2 = rng.normal(size=(4, 4))
identities_ok = [
    np.isclose(np.trace(A1 @ B1), np.trace(B1 @ A1)),
    np.isclose(np.trace(S1), lam1.sum()),
    np.isclose(np.linalg.det(S1), lam1.prod()),
    np.isclose(np.linalg.det(P1 @ P2), np.linalg.det(P1) * np.linalg.det(P2)),
]
```

</details>

### Exercise 19 · Sherman–Morrison in code
*💻 Code · ★★★*

Implement `sm_update(Ainv, u, v)` returning $(A+uv^\top)^{-1}$ from $A^{-1}$ (Exercise 11). Use it to build $(X^\top X + I)^{-1}$
**one sample at a time** (start from $A=I$, add $x_ix_i^\top$ for each row) and compare with a direct inverse.

In [ ]:
def sm_update(Ainv, u, v):
    # TODO
    return None

X_sm = rng.normal(size=(50, 4))
Ainv_running = np.eye(4)
for x in X_sm:
    upd = sm_update(Ainv_running, x, x)
    if upd is None:
        Ainv_running = None
        break
    Ainv_running = upd

check('online inverse', Ainv_running, np.linalg.inv(X_sm.T @ X_sm + np.eye(4)))

<details>
<summary><b>💡 Hint</b></summary>

`Au = Ainv @ u`, `vA = v @ Ainv`, return `Ainv - np.outer(Au, vA) / (1 + v @ Au)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each update is $O(d^2)$, so 50 updates cost $O(Nd^2)$ total, the same as forming $X^\top X$ but without ever inverting. This is recursive least squares.

```python
def sm_update(Ainv, u, v):
    Au = Ainv @ u
    vA = v @ Ainv
    return Ainv - np.outer(Au, vA) / (1.0 + v @ Au)

X_sm = rng.normal(size=(50, 4))
Ainv_running = np.eye(4)
for x in X_sm:
    Ainv_running = sm_update(Ainv_running, x, x)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Linear Algebra for ML](Linear%20Algebra%20for%20ML.md).*